# Matrix Operations

## Matrix Multiplication

Every matrix can be seen as a linear trasnformation of space, which means that it encodes how to move every vector in space to a new location. Multiplying two matricies ($A * B$) can be seen as obtaining a matrix that represents the final transformation given by applying $B$ first followed by $A$.

To make it concrete, let's work in $\mathbb{R}^2$ with standard basis vectors $\vec{i} = \begin{bmatrix}
1 \\
0
\end{bmatrix}$ and $\vec{j} = \begin{bmatrix}
0 \\
1
\end{bmatrix}$. Then take a $\vec{v} \in \mathbb{R}^2$. By definition, the first coordinate of $\vec{v}$ scales the first basis vector $\vec{i}$ and the second coordinate scales $\vec{j}$:
$$\vec{v} = v_1 * \vec{i} + v_2 * \vec{j}$$

A new matrix $A$ transforms linear space by describing where the basis vectors land after the transformation. Each column in the matrix encodes where the basis vector lands after the transformation. The first column encodes where $\vec{i}$ lands, the second column encodes where $\vec{j}$ lands, etc. Then for any arbitrary vector $\vec{v}$, to understand where it lands after the transformation, you scale the vector components by where the basis vectors land after the transformation:
$$\vec{v} = v_1 * A_{:, 1} + v_2 * A_{:, 2}$$

Matrix multiplication is then composing two transformations together to obtain a single matrix that is the result of applying one matrix after the other. To obtain this matrix, we must see where the basis vectors land after applying each transformation one after the other. Let's say we are multiplying two 2x2 matricies $A * B$. We know that $\vec{i}$ lands on $B_{:, 1}$ and $\vec{j}$ lands on $B_{:, 2}$ after applying $B$. Let's call the transformed vectors after applying $B$: $\vec{i}_B$ and $\vec{j}_B$. Then, we call $\vec{i}_A$ and $\vec{j}_A$ the new basis vectors after applying $B$ then $A$, and they land on (obtained by following the formula described above):
$$
\vec{i}_A = (i_B)_1 * A_{:, 1} + (i_B)_2 * A_{:, 2}
$$
$$
\vec{j}_B = (j_B)_1 * A_{:, 1} + (j_B)_2 * A_{:, 2}
$$

The code below generalizes this to arbitrary dimensions. It does so by iterating each column of $B$ (represents where the basis vector lands after applying $B$), and then taking each transformed $B$ basis vector and transforming it again by $A$ (takes each component of the $B$ basis vector and scales it by where the basis vectors land after applying $A$).

In [ ]:
import numpy as np

def mul(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if a.shape[1] != b.shape[0]:
        raise ValueError('Matricies have invalid shape!')

    res = np.zeros((a.shape[0], b.shape[1]))
    for i in range(b.shape[1]):
        v = b[:, i]
        for j, s in enumerate(v):
            res[:, i] += (s * a[:, j])
    return res

a, b = np.array([[1, 2, 3], [4, 5, 6]]), np.array([[1, 2], [3, 4], [5, 6]])

print("TEST CASE 1: (Success)")
print(mul(a, b))
print(np.linalg.matmul(a, b))

print("TEST CASE 2: (Invalid dimensions)")
a, b = np.array([[1, 2, 3], [4, 5, 6]]), np.array([[1, 2, 3]])
try:
    mul(a, b)
except ValueError:
    print("Got expected ValueError")

## Transpose

A transpose is defined as swapping the rows and columns of a matrix. A lot of its use is practical, ensuring dimensions align when performing matrix multiplication.

In [ ]:
def transpose(a: np.ndarray):
    res = np.zeros((a.shape[1], a.shape[0]))
    for i in range(a.shape[0]):
        for j in range(a.shape[1]):
            res[j, i] = a[i, j]
    return res

print("TEST CASE 1: (Square)")
a = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(transpose(a))
print(a.T)

print("TEST CASE 1: (Non-square)")
a = np.array([[1, 2, 3], [4, 5, 6]])
print(transpose(a))
print(a.T)

## Inverse

Earlier we described matrices as a way to linearly transform space. However, you may want to undo that transformation, and that is done by computing what's called an inverse.

We can denote the identity matrix $I$:
$$
\begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}
$$

The identity matrix leaves vectors unchanged after the transformation, which makes sense as you can see that the basis vectors $\vec{i}$ and $\vec{j}$ are unchanged in the identity matrix.

The inverse matrix of a transformation $A$ is denoted as $A^{-1}$ and satisfies the property:
$$
A^{-1}A = I
$$

This geometrically means that if you linearly transform space by matrix $A$ and transform it again by $A^{-1}$, the overall effect is to do nothing.

In [ ]:
def inverseR2(a):
    inv = a.copy()
    det = a[0, 0] * a[1, 1] - a[1, 0] * a[0, 1]
    inv[0, 0], inv[1, 1] = inv[1, 1], inv[0, 0]
    inv[1, 0], inv[0, 1] = -inv[1, 0], -inv[0, 1]
    return (1 / det) * inv

print("TEST CASE 1")
a = np.array([[1, 2], [3, 4]])
print(inverseR2(a))
print(np.linalg.inv(a))

## Determinants

Geometrically, the determinant of a matrix tells you how much a transformation scales area. Area is defined as the space taken up by a parallelotope in $\mathbf{R}^n$ given by n vectors in that space. Concretely, in $\mathbf{R}^2$ a transformation encoded by the matrix:
$
\begin{bmatrix} 2 & 0 \\ 0 & 2 \end{bmatrix}
$
would have a determinant of 4. You can see this because imagine you took the parallelogram encoded by $\vec{a} = \begin{bmatrix}
1 \\
0
\end{bmatrix}$ and $\vec{b} = \begin{bmatrix}
0 \\
1
\end{bmatrix}$. After transforming $\vec{a}$ and $\vec{b}$ by the matrix, the parallelogram given by the transformed $\vec{a}$ and $\vec{b}$ would have 4x the area (since the transformation scales the x and y axes by 2).

A determinant can be negative and that corresponds to an orientation flip. Concretely, in $\mathbf{R}^2$, $\vec{i}$ is to the right of $\vec{j}$ and you can travel counter-clockwise to get from $\vec{i}$ to $\vec{j}$. If after the transformation that no longer holds true and you have to travel clockwise to get from $\vec{i}$ to $\vec{j}$, the area has a negative determinant. This makes sense intuitively. In order for the determinant to be negative, during the transformation the basis vectors must cross over each other at some point. The determinant goes to 0 as the basis vectors get closer to each other. The determinant becomes 0 at the point they cross. The determinant becomes negative as the basis vectors go away from each other (with the flipped orientation).

In [ ]:
def determinantR2(a):
    return a[0, 0] * a[1, 1] - a[1, 0] * a[0, 1]

print("TEST CASE 1")
a = np.array([[1, 2], [3, 4]])
print(determinantR2(a))
print(np.linalg.det(a))

## Eigendecomposition

Eigendecomposition is a method of decomposing a matrix in terms of its eigenvectors and corresponding eigenvalues. The formula is as follows:
$$
A = V \Lambda V^{-1}
$$
where $V$ is a matrix of eigenvectors and $\Lambda$ is a corresponding diagonal matrix of eigenvalues.

The formula above can also be viewed in light of a change of basis. If we want to transform some vector $\vec{b}$:
$$
A \vec{b} = V \Lambda  V^{-1} \vec{b}
$$

We first convert $\vec{b}$ to "eigen-space" by applying $V^{-1}$. Then once $\vec{b}$ is represented in terms of our eigenvectors, we scale the individual components of the vector along their respective eigenvalues of $A$, and then we finally get the transformed vector back into "standard-space" by applying $V$.

Note: In order to perform eigendecomposition, the original matrix must be square with dimensions $n \times n$. Additionally, there must be $n$ linearly independent eigenvectors in order to compute the inverse of $V$. If the columns of $V$ aren't linearly independent, that means space is squished in one or more dimensions, meaning that no inverse exists.

In [ ]:
def eigendecomposition(a):
    eig = np.linalg.eig(a)
    return eig.eigenvectors, np.diag(eig.eigenvalues), np.linalg.inv(eig.eigenvectors)

print("TEST CASE 1: (Symmetric)")
a = np.array([[1, 2, 3], [2, 1, 5], [3, 5, 1]])
v, sigma, vinv = eigendecomposition(a)
print(a)
print(v @ sigma @ vinv)

print("TEST CASE 2: (Rotation)")
a = np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]])
v, sigma, vinv = eigendecomposition(a)
print(a)
print(v @ sigma @ vinv)

print("TEST CASE 3: (Custom)")
a = np.array([[1, 2, 4], [1, 2, 4], [1, 2, 4]])
try:
    v, sigma, vinv = eigendecomposition(a)
except np.linalg.LinAlgError:
    print("Expected singular matrix error")


## Singular Value Decomposition (SVD)

Unlike eigendecomposition, SVD works for any matrix because it doesn't rely on finding $n$ vectors that only get scaled (it can also handle rotation). SVD decomposes a matrix like so:
$$
A = U \Sigma V^T
$$
where $\Sigma$ is a diagonal matrix, and $U$ and $V^T$ are orthogonal matrices.

When you take a matrix such as $A$, it can be viewed as a way to transform one vector to another. To understand SVD, let's imagine we have a bunch of vectors representing the unit circle. After applying $A$ to these vectors, the vector now represents some ellipse. SVD works by first determining which input vectors (points on the original unit circle) get stretched the most when they get turned into an ellipse. To determine this, let's take a look at which vectors have the greatest squared length:
$$
\lVert Ax \rVert ^2 = (Ax)^T (Ax) = x (A^T A) x
$$

This implies that the eigenvectors of $A^T A$ are the directions that stretch $Ax$ the most. Intuitively, the reason eigenvectors of $A^T A$ get stretched the most by $A^T A$ is because eigenvectors only get stretched (not rotated), so there is no "wasted" effort going into rotation. All of the matrix transformation goes towards stretching.

Now, we can package the eigenvectors of $A^T A$ into the matrix $V$. However, we now want to figure out how to express input vectors in terms of these eigenvectors in $V$. To do that, remember how we performed eigendecomposition? We can view this in light of a change of basis. To express our input vector in terms of the stretching axes, we must multiply the input vector by $V^{-1}$. But because $V$ is orthogonal, $V^{-1} = V^T$. This is the $V^T$ part of our equation above.

Now, we need to figure out $U$ and $\Sigma$. To figure out these matricies, we can think of $U$ as a final transformation to get back to the output space. To figure out this final transformation, we can see that $Av_1 = \sigma u_1$. This is essentially saying that if you take a "preferred" direction in the input space ($v_1$) and transform it by $A$, it lands on a "preferred" direction of the output space ($u_1$). Each $u$ is unit length, and is scaled by the corresponding $\sigma$.

So effectively, what we are doing is figuring out which directions in the input space get scaled the most. Then our decomposition is telling us that if we want to figure out how any arbitrary vector $b$ gets transformed, you can first express $b$ in terms of the directions that get scaled the most ($V$), and then scale them, and then perform a final rotation to express the vector in the output space of $U$.

This is so powerful because it allows us to isolate the "energy" from the "orientation" when describing the transformation encoded by $A$.

***Additonal Notes:***

When I first learned this, I was curious why we care so much about expressing this in terms of the directions that get stretched the most. As mentioned above, one major benefit is it allows us to isolate the "energy" from the "orientation". A matrix is orthogonal if and only if it encodes a rotation or a flip. This rotation/flip is encoded in $U$ and $V$ which is why they are orthogonal. This allows the "energy" to be purely encoded by $\Sigma$.

Additionally, perpendicular eigenvectors of $A^T A$ stay perpendicular because the eigenvectors are on the principal axes of the stretch. The intuition is that vectors are pulled unevenly towards the strongest stretching directions at different rates which causes vectors to get closer together or farther apart if they don't land on the strongest stretching direction. However, if they do happen to land on the strongest direction of stretch, they just grow further in that direction and therefore remain perpendicular. This fact is important because it means that even after applying $v_1, v_2, ...$ by $A$, the output is still perpendicular. This means that $U$ also ends up being orthogonal (because of how we defined $Av_1 = \sigma u_1$). And therefore we can express $A$ as the combination of a rotation x stretch x rotation.

In [ ]:
def svd(a: np.ndarray):
    svd = np.linalg.svd(a)
    s = np.zeros(a.shape)
    k = min(a.shape)
    s[:k, :k] = np.diag(svd.S)
    return svd.U, s, svd.Vh

a = np.array([[1, 2, 3, 10], [4, 5, 6, 11], [7, 8, 9, 12]])
u, s, vh = svd(a)

print("A=")
print(a)

print("Full reconstruction")
print(u @ s @ vh)

print("Approximation")
k = 1
u_hat = u[:, :k]
s_hat = s[:k, :k]
vh_hat = vh[:k, :]
approx = u_hat @ s_hat @ vh_hat
print(approx)
error = np.linalg.norm(a - approx, 'fro')
print(f"Reconstruction error: {error}")

## Principal Component Analysis (PCA)

Principal Component Analysis is a statistical technique to represent the original data in fewer dimensions (lossy). Under the hood, PCA uses SVD to perform the dimensionality reduction.

In previous sections, we discussed a matrix $A$ as a transformation. Instead, we will switch perspectives for PCA. Now $A$ is a data matrix where each row is a sample and the columns represent features. The matrix is now viewed as a convenient way to package data rather than a representation of a linear transformation.

Let's now look at what SVD represents using this new interpretation:
$$
A = U \Sigma V^T
$$

Earlier, we said that $V$ represents the eigenvectors of the matrix $A^T A$. In the case of a data matrix, $A^T A$ is proportional to the covariance. As a reminder, covariance captures how two features change together. $A^T A$ dots each feature with each other capturing the relationship amongst features. Computing the eigenvectors of the covariance matrix gives us a list of directions that account for the most variance in the data (remember in SVD we discussed how eigenvectors of $A^T A$ are the input directions that get stretched the most). We call each of these directions a "principal component" and represents a pattern of the data. Multiplying $A$ by a specific column of $V$ can be seen as determining how much the samples of $A$ are represented by the principal component.

$U$ represents the eigenvectors of the matrix ($A A^T$). $A A^T$ is the sample-to-sample similarity matrix as we dot each sample with each other. This makes $U$ a sample map as it tells us where the samples sit relative to each other. This relative positioning helps us combine the patterns given by $V$ to recover the original samples. Concretely, the first row of $U$ tells us how much to encode the specific patterns given by $V$ to recover the first sample, the second row of $U$ helps us recover the second sample, and so forth.

$V$ and $U$ share the same eigenvalues in $\Sigma$. This is intuitvely seen as the shape of the features and the similarity of the samples have the exact same variance (as the samples are expressed in terms of the features).

Back to the original question, how can we represent data in lower dimensions as accurately as possible? Well, we want to learn the most significant patterns.

First, we center the data around $0$ which involves subtracting the mean for each feature and denote the result as $A_c$. This prevents the first learned pattern from just being a vector that points to the mean of the data. We want to learn the relationship between the features of the data not just point to the mean.

Now, we can perform SVD on the centered matrix. Remember how in the SVD section we described how SVD isolates a transformation into "energy" and "rotation". Well SVD on a data matrix does the same thing. The $\Sigma$ represents how important a pattern is. Thus, to convert our data into a lower dimension, we choose the $\Sigma$'s with the highest values and find the corresponding $U$ and $V$ vectors and pick them. So effectively, you'd choose the top-k columns of $U$ and $V$ and represent them as $U_k$ and $V_k$.

Then, to compress data you can do either:
$$
A_c * V_k
$$
or
$$
U_k * \Sigma_k
$$

The first multiplication expresses each datapoint relative to the most important principal components. $U_k * \Sigma_k$ is equivalent to the first equation based on multiplying $V$ to both sides of the equation $A = U \Sigma V^T$.

In [ ]:
from pathlib import Path

import altair as alt
import pandas as pd
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X: np.ndarray = mnist.data  # (70000, 784)
y: np.ndarray = mnist.target.astype(int)  # (70000,) labels 0-9

X_centered = X - X.mean(axis=0)
s = np.linalg.svd(X_centered, full_matrices=False)
X_hat = X_centered @ s.Vh[:2, :].T

# Stratified sample for the published chart — keeps the spec under ~250KB
# and the rendered SVG within a few MB. The site renders this with
# Vega-Lite and applies its own theme, so don't pass styling here.
rng = np.random.default_rng(seed=42)
indices = np.concatenate([
    rng.choice(np.where(y == d)[0], size=500, replace=False)
    for d in range(10)
])
rng.shuffle(indices)

df = pd.DataFrame({
    'PC 1': np.round(X_hat[indices, 0], 2),
    'PC 2': np.round(X_hat[indices, 1], 2),
    'Digit': y[indices].astype(int),
})

alt.data_transformers.disable_max_rows()

chart = (
    alt.Chart(df)
    .mark_circle(size=40, opacity=0.5)
    .encode(
        x=alt.X('PC 1:Q'),
        y=alt.Y('PC 2:Q'),
        color=alt.Color(
            'Digit:N',
            legend=alt.Legend(
                orient='right',
                direction='vertical',
                symbolType='square',
                symbolSize=400,
                symbolStrokeWidth=0,
                rowPadding=0,
            ),
        ),
    )
    .properties(width=580, height=400)
)

out = Path.home() / 'Documents/hyerra.xyz/src/content/charts/pca-mnist.json'
out.write_text(chart.to_json(indent=None))
print(f'Wrote spec ({out.stat().st_size / 1024:.1f} KB)')

chart